# Econometric Analysis of US Housing and Economic Data

This project conducts an econometric analysis of US housing and economic data using Vector Autoregression (VAR) models. The goal is to understand the impact of national macroeconomic shocks on local housing markets.

The project is divided into four notebooks:
1.  **`1_data_download.ipynb`**: Downloads and caches time series data from the Federal Reserve Economic Data (FRED) database.
2.  **`2_data_processing.ipynb`**: Loads the cached data and applies necessary transformations (e.g., differencing, detrending).
3.  **`3_econometric_analysis.ipynb`**: Performs statistical tests and fits the Structural Vector Autoregression (SVAR) model.
4.  **`4_visuals.ipynb`**: Generates and plots Impulse Response Functions (IRFs) and other visualizations.


In [4]:
# Part 1: Setup and Imports
# ==========================
# This cell imports all necessary libraries and ensures all required packages are installed.

try:
    import pandas as pd
    import pandas_datareader as pdr
    from fredapi import Fred
    import numpy as np
    import statsmodels.api as sm
    from statsmodels.tsa.api import VAR
    from statsmodels.tsa.stattools import adfuller
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from math import sqrt
except ImportError as e:
    print(f"A required package is not installed: {e}")
    print("Please run 'pip install pandas pandas_datareader fredapi numpy statsmodels matplotlib seaborn' in your terminal.")


In [5]:
# Part 2: API and Ticker Configuration
# =====================================
# This cell configures the FRED API key, date range, and the tickers for both
# national and city-level data series.

# --- Configuration ---
# IMPORTANT: Replace with your own FRED API key.
fred_key = 'c6074c01f18d7f56719e867c8815863e' 
fred = Fred(api_key=fred_key)

# Set the desired date range for the analysis
start_date = '1991-01-01'
end_date = '2007-12-31'

# --- Ticker Definitions ---
# Define the 3 common national tickers used across all analyses
common_national_tickers = {
    'indpro': 'INDPRO',      # Industrial Production
    'cpi': 'CPIAUCSL',       # Consumer Price Index
    'fedfunds': 'FEDFUNDS',  # Federal Funds Rate
}

# Define the area-specific tickers for House Price Index (HPI), Unemployment, and Building Permits
area_specific_tickers = {
    'National': {'hpi': 'CSUSHPINSA', 'unemp': 'UNRATE', 'permits': 'PERMIT'},
    'SF': {'hpi': 'SFXRSA', 'unemp': 'SANF806UR', 'permits': 'SANF806BPPRIVSA'},
    'DET': {'hpi': 'DEXRSA', 'unemp': 'DETR826UR', 'permits': 'DETR826BPPRIVSA'},
    'ATL': {'hpi': 'ATXRSA', 'unemp': 'ATLA013UR', 'permits': 'ATLA013BPPRIVSA'},
    'LV': {'hpi': 'LVXRSA', 'unemp': 'LASV832UR', 'permits': 'LASV832BPPRIVSA'}
}


In [6]:
# Part 3: Data Fetching and Caching
# ==================================
# This cell handles downloading the data for each area, combining national and
# local series, and caching the results to disk to prevent re-downloading.

data = {}
cache_dir = 'data_cache'
os.makedirs(cache_dir, exist_ok=True)

# --- Fetch Common National Data ---
# Fetches the national-level data that is common to all areas. This is done
# once to avoid redundant API calls.
print("Fetching common national data series...")
national_df_common = pd.DataFrame()
for name, ticker in common_national_tickers.items():
    try:
        series = fred.get_series(ticker, start_date, end_date)
        national_df_common[name] = series
    except Exception as e:
        print(f"Could not fetch common national series {name} ({ticker}): {e}")
        continue # Continue to the next ticker

# --- Fetch Area-Specific Data and Combine ---
# Loop through each area defined in the configuration, fetch its specific data,
# combine it with the national data, and cache it.
for area, specific_tickers in area_specific_tickers.items():
    cache_path = os.path.join(cache_dir, f'{area}_data.csv')
    
    try:
        # If a cached file exists, load it directly to save time
        if os.path.exists(cache_path):
            print(f"Loading cached data for {area} from {cache_path}...")
            area_df = pd.read_csv(cache_path, index_col='date', parse_dates=True)
        
        # If no cached file is found, fetch the data from the FRED API
        else:
            print(f"Fetching data for {area} from FRED API...")
            
            # Fetch the 3 area-specific series
            area_df_specific = pd.DataFrame()
            for name, ticker in specific_tickers.items():
                try:
                    series = fred.get_series(ticker, start_date, end_date)
                    area_df_specific[name] = series
                except Exception as e:
                    print(f"Could not fetch specific series {name} ({ticker}) for {area}: {e}")
                    continue

            # Combine the common national data with the area-specific data
            area_df = pd.concat([national_df_common, area_df_specific], axis=1)
            
            # Clean the combined DataFrame by dropping rows with any missing values
            area_df.dropna(inplace=True)
            area_df.index.name = 'date'
            
            # Save the cleaned DataFrame to a CSV file for future use
            area_df.to_csv(cache_path)
            print(f"Saved and cached data for {area} to {cache_path}")

        # Store the final DataFrame in the 'data' dictionary
        data[area] = area_df

    except Exception as e:
        print(f"An error occurred while processing data for {area}: {e}")

print("\\nAll data has been successfully fetched and loaded.")

# --- Verification ---
# Display the first few rows of one of the DataFrames to verify the process
if 'SF' in data:
    print("\\nSample of data for SF:")
    print(data['SF'].head())


Fetching common national data series...
Loading cached data for National from data_cache\National_data.csv...
Loading cached data for SF from data_cache\SF_data.csv...
Loading cached data for DET from data_cache\DET_data.csv...
Loading cached data for ATL from data_cache\ATL_data.csv...
Loading cached data for LV from data_cache\LV_data.csv...
\nAll data has been successfully fetched and loaded.
\nSample of data for SF:
             indpro    cpi  fedfunds        hpi  unemp      permits
date                                                               
1991-01-01  61.0842  134.7      6.91  71.935495    4.6  2037.040992
1991-02-01  60.6387  134.8      6.25  71.126390    4.7   585.889721
1991-03-01  60.2973  134.8      6.12  70.608208    4.9   835.829789
1991-04-01  60.4524  135.1      5.91  69.909268    5.0   789.687733
1991-05-01  61.0587  135.6      5.78  70.010559    5.1   625.659862
